# Example

This notebook demonstrates how to use Wags-LLM using the `BedrockClaudeJsonClient`.

In [ ]:
import logging
import sys

from pydantic import BaseModel, ConfigDict

from wags_llm.cache import InMemoryCache
from wags_llm.client.bedrock import BedrockClaudeJsonClient
from wags_llm.registry.base import Registry
from wags_llm.services.structured_task import StructuredTaskRunner
from wags_llm.templates.base import PromptTemplate

logging.basicConfig(
    stream=sys.stdout,
    level=logging.WARNING,
    format="%(name)s - %(levelname)s - %(message)s",
)
logging.getLogger("wags_llm").setLevel(logging.DEBUG)

Let's pretend we want to find the MONDO identifier for a given free-text label. We want a simple response to be `mondo_id` as a string.

In [ ]:
class MyPrompt(PromptTemplate):
    name = "mondo_id_classification"
    version = "v1"

    def build_system_prompt(self):
        return "Given free-text label, get the associated MONDO identifier."

    def build_user_prompt(self, payload):
        return f"Input:\n{payload['text']}"


class Result(BaseModel):
    model_config = ConfigDict(extra="forbid")  # Required

    mondo_id: str

We want to provide a JSON schema for the model response to conform to.

In [3]:
json_schema = Result.model_json_schema()
json_schema

{'additionalProperties': False,
 'properties': {'mondo_id': {'title': 'Mondo Id', 'type': 'string'}},
 'required': ['mondo_id'],
 'title': 'Result',
 'type': 'object'}

We will be using Claude Sonnet 4.6 and will demonstrate how to use a cache (optional).

In [ ]:
client = BedrockClaudeJsonClient(
    model_id="us.anthropic.claude-sonnet-4-6",
    region_name="us-east-1",
    profile_name="dev-account",
)

registry = Registry()
registry.register(MyPrompt())

service = StructuredTaskRunner(
    client=client,
    registry=registry,
    cache=InMemoryCache(),
)

wags_llm.client.bedrock - DEBUG - BedrockClaudeJsonClient config: model_id='us.anthropic.claude-sonnet-4-6', region_name='us-east-1', profile_name='dev-account', max_tokens=300, temperature=0.000000
wags_llm.client.bedrock - INFO - BedrockClaudeJsonClient successfully initialized for model_id='us.anthropic.claude-sonnet-4-6'
wags_llm.prompts.registry - DEBUG - Registering prompt: name='mondo_id_classification', version='v1'


In [ ]:
result = service.execute_prompt(
    prompt_name="mondo_id_classification",
    prompt_version="v1",
    payload={"text": "melanoma"},
    response_model=Result,
)
result

wags_llm.services.structured_task - DEBUG - Cache lookup using key='36ed4e9296d226033f6569fc34e8e4ddf36300120175447ac11e373873fc8be5' (for cache_payload={'payload': {'text': 'melanoma'}, 'model': 'us.anthropic.claude-sonnet-4-6', 'prompt_name': 'mondo_id_classification', 'prompt_version': 'v1'})
wags_llm.cache.in_memory - DEBUG - Cache miss for cache key='36ed4e9296d226033f6569fc34e8e4ddf36300120175447ac11e373873fc8be5'
wags_llm.client.bedrock - DEBUG - Bedrock Claude usage={'inputTokens': 190, 'outputTokens': 15, 'totalTokens': 205, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
wags_llm.client.bedrock - DEBUG - Bedrock Claude metrics={'latencyMs': 2159}
wags_llm.client.bedrock - DEBUG - Bedrock Claude content=[{'text': '{"mondo_id":"MONDO:0005105"}'}]


Result(mondo_id='MONDO:0005105')

We can see that the cache works when running the same cell again

In [ ]:
result = service.execute_prompt(
    prompt_name="mondo_id_classification",
    prompt_version="v1",
    payload={"text": "melanoma"},
    response_model=Result,
)
result

wags_llm.services.structured_task - DEBUG - Cache lookup using key='36ed4e9296d226033f6569fc34e8e4ddf36300120175447ac11e373873fc8be5' (for cache_payload={'payload': {'text': 'melanoma'}, 'model': 'us.anthropic.claude-sonnet-4-6', 'prompt_name': 'mondo_id_classification', 'prompt_version': 'v1'})
wags_llm.cache.in_memory - DEBUG - Cache hit for cache key='36ed4e9296d226033f6569fc34e8e4ddf36300120175447ac11e373873fc8be5'


Result(mondo_id='MONDO:0005105')